<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/different_types_of_music_genres_using_audio_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environmental Sound Classification
This project focuses on classifying environmental sounds (Dog Bark, Siren, Rain) using a Convolutional Neural Network (CNN) trained on Mel-Spectrogram features.

In [2]:
import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Configuration
IMG_HEIGHT = 40
IMG_WIDTH = 174
CLASSES = ['dog_bark', 'siren', 'rain']

print("Libraries imported and configuration set.")

Libraries imported and configuration set.


In [3]:
def extract_features(file_path):
    try:
        # Load audio
        audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')
        # Generate MFCCs
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        # Pad to fixed width
        if mfccs.shape[1] < IMG_WIDTH:
            pad_width = IMG_WIDTH - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :IMG_WIDTH]
        return mfccs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

## Define the CNN Architecture
We treat the Mel-Spectrogram as a single-channel image for classification.

In [4]:
model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 38, 172, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 17, 84, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 21504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,752,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,771,843 (10.57 MB)

 Trainable params: 2,771,843 (10.57 MB)

 Non-trainable params: 0 (0.00 B)

## 3. Data Simulation and Model Training
Since we are in a sandbox environment without a physical audio dataset, we will generate synthetic Mel-Spectrogram data to demonstrate the training loop.

In [7]:
# Generate 300 synthetic samples
num_samples = 300
X = np.random.rand(num_samples, IMG_HEIGHT, IMG_WIDTH, 1)
y = np.random.randint(0, len(CLASSES), num_samples)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Training data shape: (240, 40, 174, 1)
Testing data shape: (60, 40, 174, 1)
Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 287ms/step - accuracy: 0.3458 - loss: 1.1039 - val_accuracy: 0.2667 - val_loss: 1.1061
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 274ms/step - accuracy: 0.3542 - loss: 1.0979 - val_accuracy: 0.2667 - val_loss: 1.1051
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 266ms/step - accuracy: 0.3542 - loss: 1.0988 - val_accuracy: 0.2667 - val_loss: 1.1054
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 439ms/step - accuracy: 0.3542 - loss: 1.0969 - val_accuracy: 0.2667 - val_loss: 1.1054
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 252ms/step - accuracy: 0.3542 - loss: 1.0976 - val_accuracy: 0.2667 - val_loss: 1.1055
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 252ms/step - accuracy: 0.3542 - loss: 1.0976 - val_accuracy: 0.2667 - val_loss: 1.1060
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 251ms/step - accuracy: 0.3542 - loss: 1.0975 - val_accuracy: 0.2667 - val_loss: 1.1062
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 280

## 3. Data Simulation and Model Training
Since we are in a sandbox environment without a physical audio dataset, we will generate synthetic Mel-Spectrogram data to demonstrate the training loop.

In [6]:
# Generate 300 synthetic samples
num_samples = 300
X = np.random.rand(num_samples, IMG_HEIGHT, IMG_WIDTH, 1)
y = np.random.randint(0, len(CLASSES), num_samples)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Training data shape: (240, 40, 174, 1)
Testing data shape: (60, 40, 174, 1)
Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 356ms/step - accuracy: 0.3750 - loss: 1.1055 - val_accuracy: 0.4000 - val_loss: 1.0972
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 281ms/step - accuracy: 0.3417 - loss: 1.0983 - val_accuracy: 0.4000 - val_loss: 1.0975
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step - accuracy: 0.3708 - loss: 1.0977 - val_accuracy: 0.4000 - val_loss: 1.0972
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 284ms/step - accuracy: 0.3750 - loss: 1.0974 - val_accuracy: 0.4000 - val_loss: 1.0967
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 410ms/step - accuracy: 0.3833 - loss: 1.0967 - val_accuracy: 0.4000 - val_loss: 1.0963
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 256ms/step - accuracy: 0.3750 - loss: 1.0966 - val_accuracy: 0.4000 - val_loss: 1.0937
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 258ms/step - accuracy: 0.3583 - loss: 1.0951 - val_accuracy: 0.4000 - val_loss: 1.0963
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 251

## Generate Synthetic Data and Train
Since we don't have the physical audio files in this environment yet, we will generate synthetic feature data to demonstrate the training pipeline.

In [5]:
# Generating 300 samples of synthetic data
num_samples = 300
X = np.random.rand(num_samples, IMG_HEIGHT, IMG_WIDTH, 1)
y = np.random.randint(0, len(CLASSES), num_samples)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Training data shape: (240, 40, 174, 1)
Testing data shape: (60, 40, 174, 1)
Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 483ms/step - accuracy: 0.3125 - loss: 1.4721 - val_accuracy: 0.3833 - val_loss: 1.0955
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 294ms/step - accuracy: 0.3417 - loss: 1.0969 - val_accuracy: 0.3333 - val_loss: 1.0992
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 279ms/step - accuracy: 0.4125 - loss: 1.0975 - val_accuracy: 0.3167 - val_loss: 1.0994
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 265ms/step - accuracy: 0.3958 - loss: 1.0983 - val_accuracy: 0.3833 - val_loss: 1.0950
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 281ms/step - accuracy: 0.3292 - loss: 1.0971 - val_accuracy: 0.3333 - val_loss: 1.0951
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 461ms/step - accuracy: 0.3750 - loss: 1.0951 - val_accuracy: 0.3333 - val_loss: 1.0983
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 278ms/step - accuracy: 0.3708 - loss: 1.0965 - val_accuracy: 0.3333 - val_loss: 1.1010
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 290